<a href="https://colab.research.google.com/github/MichalSlowakiewicz/Visual-Recognition/blob/master/HW2_circles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 2: circle detection
*(Visual Recognition 2025/26)*

You are given [fundus photographs](https://en.wikipedia.org/wiki/Fundus_photography) (images of the rear of the human eye).
Your task is to find the circular outline of the image, quickly and robustly.
This is nontrivial, because of all kinds of artifacts (noise, over- and underexposure, obscured or cut-out parts of the image, etc.)
Run the notebook to see examples near the end.

Implement two algorithms:
* A. The first should follow a prescribed approach described below (a variant of the Hough transform). Here you may *not* use existing circle detection implementations like `cv2.HoughCircles`.
* B. The second can be any method of your choice. This can be a significant modification of 1. (more than just changing parameters), or a completely different method: some simple sweeping heuristic, a machine learning model, something based on existing implementations like `cv2.HoughCircles`, etc. You may use a GPU (but this homework can be done without GPUs).

Both algorithms will be graded in the same way, for 75% and 25% of the grade respectively.
Both will be graded on a hidden test set based on the following:
* success rate (how many images are correctly detected, where we define *correct* as IoU > 0.85),
* accuracy: mean of `clip(IoU, min=0.85, max=0.98)` (all incorrect detections are treated equally, IoU above 0.98 is irrelevant),
* speed: the running time of the algorithm on a Colab CPU *must* be below 2s per image on average (you can assume that a warm-up run was performed before measuring times). Be as fast as possible, but prioritize accuracy.

### Algorithm A requirements
For algorithm A, the approach should be as follows:
* Make an array of scores for a range of center coordinates and radii (cy, cx, r).
* For each candidate circle (cy, cx, r), the score should be the sum of contributions from a few points on the circle.
* The contribution of each image point to a candidate circle should be based on the image gradient at that point: consider the gradient vector $(\frac{\partial \texttt{img}}{\partial y}, \frac{\partial \texttt{img}}{\partial x})$. These can be computed with a Sobel or Scharr filter ([`cv2.Scharr()`](https://docs.opencv.org/4.x/d5/d0f/tutorial_py_gradients.html)).
* The contribution of each image point should be higher when the gradient vector points towards the candidate center and when the gradient is larger.
* Return the (cy, cx, r) with the highest score.
* For best results, consider: how to strengthen the impact of different factors (gradient angle vs gradient norm); how to handle noise in the image and gradient; how to avoid large gradients in bright parts of the image dwarfing the darker borders.
* For speed, consider: in what order to perform operations; how to minimize processed data given that we don't need to be pixel-perfect.
* Make the algorithm iterative (with 2 or 3 steps): first run it to obtain a rough estimate, then run it again on a finer choice of candidate circle parameters.
* Use *numba* to make the algorithm fast on CPU.
  
Please make sure to ***only modify sections `Algorithm A` and `Algorithm B`*** of the notebook.
You may add imports (available on Colab), functions, shared code, etc. there, but make sure the notebook can be executed unattended from top to bottom without errors in a reasonable time,
resulting in functions `find_circle_A` and `find_circle_B` being defined (for automated tests).
If you add any heavy computations like evaluations, put them in extra sections at the end.

### Numba
Numba is a just-in-time compiler that is easy to use with numpy code. See [docs](https://numba.readthedocs.io/en/stable/user/5minguide.html). In short, this is all you need to know:
* Use the `@numba.njit` decorator on a function: it will be automatically compiled on first usage: this takes a while, but the resulting function is usually much faster.
* Not all code can be used in functions JITed with numba:
    * Limit usage to int/float and numpy.ndarray types (especially function arguments).
    * Most numpy functions are available, but a few will throw errors: you may need to replace them with more elementary implementations.
* For this reason, you will probably need to limit numba to a helper function that only does the heavy computation.
* You often do not need to vectorize your code at all, since numba's JIT compiler optimizes simple nested `for` loops very well.
    
### OpenCV
OpenCV is a popular computer vision library with many functions from basic image processing, edge detection, to 3D waypoint estimation and more.
The Python package (`cv2`) is essentially just bindings to the C++ library, so it is very fast, but also a quite clunky to use at times.
Their documentation focuses on the C++ API: e.g. output arrays are often function arguments in C++, but returned values in Python.
When in doubt, check the examples/tutorials in the documentation.

## Imports

In [ ]:
import json
import math
import time
from collections.abc import Callable, Iterable, Sequence
from functools import partial
from pathlib import Path
from typing import TypedDict, cast

import cv2
import ipywidgets
import matplotlib.pyplot as plt
import numba
import numpy as np
import pandas as pd
import PIL.Image
import PIL.ImageDraw
from IPython.display import display
from tqdm.contrib.concurrent import process_map

## Dataset
The dataset is based on a [Kaggle dataset](https://www.kaggle.com/competitions/diabetic-retinopathy-detection/overview) of fundus photographs for diagnosing diabetic retinopathy.
However, it is much smaller, downscaled, and modified to exaggerate the artifacts according to a difficulty parameter 0..90.

* It is split into three parts: 1000 train images, 100 validation images, and a lot of test images (hidden).
* You are given ground-truths for the small validation set only.
* In each part, you have subfolders of equal size for each difficulty level (0, 10, 20, ..., 90).
    They are only here to help you estimate the performance of your algorithms.
    Difficulty 0 are unmodified images – already quite challenging, but all solvable.
    As difficulty increases, a few images may be even impossible to decide – don't expect 100% success rate.
* The test set follows the same distribution.
* You may assume that all images satisfy the following (where $H,W$ are image dimensions, $r$ is the radius in pixels, $\langle cy, cx \rangle$ is the center in pixel coordinates):
    * the center is comfortably within image bounds: $0.3\,H \leq cy \leq 0.7\,H$ and $0.3\,W \leq cx \leq 0.7\,W$
    * each corner of the image is at least $1.2r$ away from the center.
    * $0.25\,\max(H, W) \leq r \leq 0.75\,\min(H, W)$
    


In [1]:
%%bash
mkdir -p data/
if [ ! -d "data/circle/train" ]; then
    # 50 MiB disk space
    wget -q 'https://www.mimuw.edu.pl/~mw290715/upload/circle.tar.gz' -O data/circle.tar.gz
    cd data/
    tar -xf circle.tar.gz
    cd -
fi

/content


In [2]:
IMG_DIR = Path("data/circle/")
train_set = sorted((IMG_DIR / "train").rglob("*.webp"))
val_set = sorted((IMG_DIR / "val").rglob("*.webp"))
assert len(train_set) == 1000 and len(val_set) == 100, (
    "Bad number of images in train/val sets."
)

NameError: name 'Path' is not defined

In [ ]:
class Result(TypedDict):
    path: str  # Relative to IMG_DIR. Always an RGB webp image.
    h: int  # Image height.
    w: int  # Image width.
    cy: int  # Y coordinate of circle center.
    cx: int  # X coordinate of circle center.
    r: int  # Radius of the circle.
    duration: float  # Time taken to find the circle, in seconds.

In [ ]:
GROUND_TRUTHS: dict[Path, Result] = {}

for part in ("train", "val", "test"):
    if not (IMG_DIR / f"{part}.json").exists():
        continue
    with (IMG_DIR / f"{part}.json").open() as f:
        for result in json.load(f):
            GROUND_TRUTHS[IMG_DIR / result["path"]] = result

df = pd.DataFrame.from_dict(GROUND_TRUTHS, orient="index")
df["difficulty"] = df.index.map(lambda p: p.parent.name)
df.head()

## Evaluation utils

In [ ]:
def run_algorithm(
    img_path: Path,
    algo: Callable[[PIL.Image.Image], tuple[int, int, int]],
) -> Result:
    pil_img = PIL.Image.open(img_path)
    start_time = time.perf_counter()
    cy, cx, r = algo(pil_img)
    duration = time.perf_counter() - start_time
    return {
        "path": str(img_path.relative_to(IMG_DIR)),
        "h": pil_img.height,
        "w": pil_img.width,
        "cy": cy,
        "cx": cx,
        "r": r,
        "duration": round(duration, 4),
    }

In [ ]:
class ResultWithComparison(Result):
    c_dist: float  # Distance from predicted to GT center, as ratio of image width.
    r_diff: float  # Difference of predicted minus GT radius (possibly negative), as ratio of GT radius.
    iou: float  # Intersection over Union of predicted vs GT disc. Between 0 and 1.


def get_iou(pred: tuple[int, int, int], gt: tuple[int, int, int]) -> float:
    """IoU of two circles, each specified as (cy, cx, r)."""
    pred_cy, pred_cx, pred_r = pred
    gt_cy, gt_cx, gt_r = gt

    d = math.hypot(pred_cy - gt_cy, pred_cx - gt_cx)
    r1, r2 = max(pred_r, gt_r), min(pred_r, gt_r)

    if d >= pred_r + gt_r:
        return 0.0
    elif d <= r1 - r2:
        return math.pi * (r2**2) / (math.pi * (r1**2))

    # https://mathworld.wolfram.com/Circle-CircleIntersection.html
    d1 = (r1**2 - r2**2 + d**2) / (2 * d)
    d2 = d - d1
    t = 0.5 * math.sqrt((-d + r1 + r2) * (d + r1 - r2) * (d - r1 + r2) * (d + r1 + r2))
    intersection_area = r2**2 * math.acos(d2 / r2) + r1**2 * math.acos(d1 / r1) - t
    union_area = math.pi * (pred_r**2 + gt_r**2) - intersection_area
    return intersection_area / union_area


def run_and_compare(
    img_path: Path, algo: Callable[[PIL.Image.Image], tuple[int, int, int]]
) -> ResultWithComparison:
    pred = run_algorithm(img_path, algo=algo)
    if img_path not in GROUND_TRUTHS:
        return {
            **pred,
            "c_dist": float("nan"),
            "r_diff": float("nan"),
            "iou": float("nan"),
        }
    gt = GROUND_TRUTHS[img_path]
    c_dist = np.hypot(pred["cy"] - gt["cy"], pred["cx"] - gt["cx"]) / gt["w"]
    r_diff = (pred["r"] - gt["r"]) / gt["r"]
    iou = get_iou((pred["cy"], pred["cx"], pred["r"]), (gt["cy"], gt["cx"], gt["r"]))
    return {
        **pred,
        "c_dist": round(float(c_dist), 4),
        "r_diff": round(float(r_diff), 4),
        "iou": round(float(iou), 4),
    }


def run_all(
    img_paths: Iterable[Path],
    algo: Callable[[PIL.Image.Image], tuple[int, int, int]],
    num_workers: int = 2,
) -> list[ResultWithComparison]:
    return process_map(
        partial(run_and_compare, algo=algo), img_paths, max_workers=num_workers
    )

## Visualization

In [ ]:
def show_result(
    result: Result, gt: Result | None = None, max_height: int = 512
) -> PIL.Image.Image:
    print(result)
    pil_img = PIL.Image.open(IMG_DIR / result["path"])
    draw = PIL.ImageDraw.Draw(pil_img)

    # Draw ground truth in red, behind, with a small circle at the center.
    if gt:
        xy = (gt["cx"], gt["cy"])
        draw.circle(xy=xy, radius=gt["r"], outline="red", width=2)
        draw.circle(xy=xy, radius=30, outline="red", width=2)

    # Draw this result in blue, with a small circle at the center.
    xy = (result["cx"], result["cy"])
    draw.circle(xy=xy, radius=result["r"], outline="blue", width=3)
    draw.circle(xy=xy, radius=30, outline="blue", width=3)

    downscale = max(1, math.ceil(pil_img.height / max_height))
    return pil_img.reduce(downscale)


def show_results(
    img_paths: Sequence[Path] | pd.Index,
    algo: Callable[[PIL.Image.Image], tuple[int, int, int]],
    init_i: int = 0,
) -> None:
    if isinstance(img_paths, pd.Index):
        img_paths = cast(Sequence[Path], img_paths)

    def f(i: int) -> None:
        print(f"Image {i + 1}/{len(img_paths)}: {img_paths[i].name}")
        if img_paths[i] in GROUND_TRUTHS:
            result = run_and_compare(img_paths[i], algo=algo)
            display(show_result(result, GROUND_TRUTHS[img_paths[i]]))
        else:
            result = run_algorithm(img_paths[i], algo=algo)
            display(show_result(result))

    ipywidgets.interact(
        f, i=ipywidgets.IntSlider(min=0, max=len(img_paths) - 1, value=init_i)
    )

In [ ]:
def find_circle_baseline(pil_img: PIL.Image.Image) -> tuple[int, int, int]:
    cy, cx, r = (
        pil_img.height // 2,
        pil_img.width // 2,
        min(pil_img.height, pil_img.width) // 2,
    )
    return cy, cx, r

In [ ]:
results = run_all(val_set, algo=find_circle_baseline)
results_df = pd.DataFrame(results)
results_df["difficulty"] = results_df["path"].map(lambda p: p.split("/")[1])
results_df["index"] = results_df["path"].apply(lambda p: IMG_DIR / p)
results_df = results_df.drop_duplicates(subset="index", keep="last").set_index("index")
results_df

In [ ]:
success_rate = (results_df["iou"] >= 0.85).mean()
accuracy = results_df["iou"].clip(0.85, 0.98).mean()
print(f"Success Rate: {success_rate:.1%}, Accuracy: {accuracy:.2%}")

The visualization shows:
* the ground truth in red;
* the prediction of a given algorithm in blue.
* for both: a smaller circle around the center.

In [ ]:
# ordering = train_set
ordering = df.index
# ordering = (df["r"] / df["w"]).sort_values(ascending=True).index
# ordering = results_df.sort_values("iou", ascending=True).index
show_results(ordering, algo=find_circle_baseline)

## Algorithm A

In [ ]:
def find_circle_A(pil_img: PIL.Image.Image) -> tuple[int, int, int]:
    cy, cx, r = find_circle_baseline(pil_img)
    return cy, cx, r

## Algorithm B

In [ ]:
def find_circle_B(pil_img: PIL.Image.Image) -> tuple[int, int, int]:
    cy, cx, r = find_circle_baseline(pil_img)
    return cy, cx, r